In [ ]:
%%bash
set -euxo pipefail
export DEBIAN_FRONTEND=noninteractive

echo "[STEP 1/5] base packages $(date -u)"
apt-get update
apt-get install -y --no-install-recommends \
  ca-certificates git git-lfs curl unzip zip expect \
  openjdk-17-jdk-headless python3-pip \
  build-essential cmake ninja-build ccache rsync \
  autoconf automake libtool pkg-config gperf perl yasm nasm

python3 -m pip install --upgrade pip setuptools wheel
python3 -m pip install aqtinstall==3.2.1

echo "[DONE] step 1/5 $(date -u)"


In [ ]:
%%bash
set -euxo pipefail
export ANDROID_SDK_ROOT=/content/android-sdk

echo "[STEP 2/5] android cmdline-tools $(date -u)"
mkdir -p "$ANDROID_SDK_ROOT/cmdline-tools"
if [ ! -x "$ANDROID_SDK_ROOT/cmdline-tools/latest/bin/sdkmanager" ]; then
  curl -fL --retry 5 --retry-delay 5 \
    https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip \
    -o /tmp/android-cmdline-tools.zip
  unzip -q /tmp/android-cmdline-tools.zip -d "$ANDROID_SDK_ROOT/cmdline-tools"
  rm -rf "$ANDROID_SDK_ROOT/cmdline-tools/latest"
  mv "$ANDROID_SDK_ROOT/cmdline-tools/cmdline-tools" "$ANDROID_SDK_ROOT/cmdline-tools/latest"
fi

echo "[DONE] step 2/5 $(date -u)"


In [ ]:
%%bash
set -euxo pipefail
export ANDROID_SDK_ROOT=/content/android-sdk

echo "[STEP 3/5] accept sdk licenses $(date -u)"
set +o pipefail
timeout 900 bash -lc 'yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --sdk_root=/content/android-sdk --licenses --verbose' \
  > /tmp/sdk-licenses.log 2>&1
licenses_rc=$?
set -o pipefail
tail -n 120 /tmp/sdk-licenses.log || true
if [ "$licenses_rc" -eq 124 ]; then
  echo "sdkmanager --licenses timeout (15 min)"
  exit 124
fi
if [ "$licenses_rc" -ne 0 ]; then
  echo "sdkmanager --licenses failed: $licenses_rc"
  exit "$licenses_rc"
fi

echo "[DONE] step 3/5 $(date -u)"


In [ ]:
%%bash
set -euxo pipefail
export ANDROID_SDK_ROOT=/content/android-sdk

echo "[STEP 4/5] install sdk packages $(date -u)"
set +o pipefail
yes | "$ANDROID_SDK_ROOT/cmdline-tools/latest/bin/sdkmanager" --sdk_root="$ANDROID_SDK_ROOT" \
  "platform-tools" "platforms;android-34" "build-tools;34.0.0" \
  "ndk;26.1.10909125" "cmake;3.22.1"
sdk_rc=${PIPESTATUS[1]}
set -o pipefail
if [ "$sdk_rc" -ne 0 ]; then
  echo "sdkmanager packages failed: $sdk_rc"
  exit "$sdk_rc"
fi

echo "[DONE] step 4/5 $(date -u)"


In [ ]:
%%bash
set -euxo pipefail

echo "[STEP 5/5] write env $(date -u)"
cat > /content/ayumobile_env.sh <<'ENVEOF'
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export ANDROID_SDK_ROOT=/content/android-sdk
export ANDROID_HOME=$ANDROID_SDK_ROOT
export ANDROID_NDK_ROOT=$ANDROID_SDK_ROOT/ndk/26.1.10909125
export PATH=$JAVA_HOME/bin:$ANDROID_SDK_ROOT/platform-tools:$ANDROID_SDK_ROOT/cmdline-tools/latest/bin:$PATH
ENVEOF

echo "[DONE] step 5/5 $(date -u)"


In [ ]:
%%bash
set -euxo pipefail
source /content/ayumobile_env.sh

echo "[QT HOST] start $(date -u)"
QT_ROOT=/content/qt
QT_VER=6.8.3
mkdir -p "$QT_ROOT"
if [ ! -d "$QT_ROOT/$QT_VER/gcc_64" ]; then
  python3 -u -m aqt install-qt -O "$QT_ROOT" linux desktop "$QT_VER" linux_gcc_64
fi
echo "[QT HOST] done $(date -u)"


In [ ]:
%%bash
set -euxo pipefail
source /content/ayumobile_env.sh

echo "[QT ANDROID] start $(date -u)"
QT_ROOT=/content/qt
QT_VER=6.8.3
if [ ! -d "$QT_ROOT/$QT_VER/android_arm64_v8a" ]; then
  python3 -u -m aqt install-qt -O "$QT_ROOT" linux android "$QT_VER" android_arm64_v8a
fi
echo "[QT ANDROID] done $(date -u)"


In [ ]:
%%bash
set -euxo pipefail
source /content/ayumobile_env.sh

QT_ROOT=/content/qt
QT_VER=6.8.3
export QT_HOST_PATH=$QT_ROOT/$QT_VER/gcc_64
export QT_ANDROID_PATH=$QT_ROOT/$QT_VER/android_arm64_v8a
export PATH=$QT_HOST_PATH/bin:$PATH

test -f "$QT_ANDROID_PATH/lib/cmake/Qt6/qt.toolchain.cmake"
test -f "$QT_ANDROID_PATH/lib/cmake/Qt6/Qt6Config.cmake"

cd /content
if [ ! -d ayumobile ]; then
  env -u HTTP_PROXY -u HTTPS_PROXY -u ALL_PROXY -u http_proxy -u https_proxy -u all_proxy     git clone --recursive -b dev https://github.com/Perdonus/ayumobile.git
fi

cd /content/ayumobile
env -u HTTP_PROXY -u HTTPS_PROXY -u ALL_PROXY -u http_proxy -u https_proxy -u all_proxy   git fetch origin dev
env -u HTTP_PROXY -u HTTPS_PROXY -u ALL_PROXY -u http_proxy -u https_proxy -u all_proxy   git reset --hard origin/dev

git submodule sync --recursive
git submodule update --init --recursive
chmod +x scripts/apply-android-submodule-patches.sh
./scripts/apply-android-submodule-patches.sh

chmod +x scripts/prepare-android-deps.sh
DEPS_ROOT=/content/android-deps SRC_ROOT=/content/android-deps-src ANDROID_API=29 ./scripts/prepare-android-deps.sh

DEPS_ROOT=/content/android-deps
DEPS_PREFIX=$DEPS_ROOT/prefix-arm64
FFMPEG_PREFIX=$DEPS_ROOT/ffmpeg-android-arm64

test -f "$DEPS_PREFIX/lib/libtde2e.a"
test -f "$DEPS_PREFIX/lib/libtg_owt.a"
test -f "$FFMPEG_PREFIX/lib/libavcodec.a"

export PKG_CONFIG_PATH=$DEPS_PREFIX/lib/pkgconfig:$FFMPEG_PREFIX/lib/pkgconfig
export PKG_CONFIG_LIBDIR=$PKG_CONFIG_PATH

rm -rf out/android-arm64-native

echo "[BUILD] configure $(date -u)"
cmake -S . -B out/android-arm64-native -GNinja   -DCMAKE_BUILD_TYPE=Release   -DCMAKE_TOOLCHAIN_FILE="$QT_ANDROID_PATH/lib/cmake/Qt6/qt.toolchain.cmake"   -DCMAKE_SYSTEM_NAME=Android   -DCMAKE_PREFIX_PATH="$DEPS_PREFIX;$QT_ANDROID_PATH"   -DQT_HOST_PATH="$QT_HOST_PATH"   -DANDROID_SDK_ROOT="$ANDROID_SDK_ROOT"   -DANDROID_NDK_ROOT="$ANDROID_NDK_ROOT"   -DANDROID_ABI=arm64-v8a   -DANDROID_PLATFORM=android-29   -DDESKTOP_APP_USE_PACKAGED=ON   -DDESKTOP_APP_SPECIAL_TARGET=android   -DJPEG_INCLUDE_DIR="$DEPS_PREFIX/include"   -DJPEG_LIBRARY="$DEPS_PREFIX/lib/libjpeg.a"   -DOpenAL_DIR="$DEPS_PREFIX/lib/cmake/OpenAL"   -Dtde2e_DIR="$DEPS_PREFIX/lib/cmake/tde2e"   -Dtg_owt_DIR="$DEPS_PREFIX/lib/cmake/tg_owt"   -DOPENSSL_ROOT_DIR="$DEPS_PREFIX"   -DOPENSSL_INCLUDE_DIR="$DEPS_PREFIX/include"   -DOPENSSL_SSL_LIBRARY="$DEPS_PREFIX/lib/libssl.a"   -DOPENSSL_CRYPTO_LIBRARY="$DEPS_PREFIX/lib/libcrypto.a"   -DTDESKTOP_API_ID=17349   -DTDESKTOP_API_HASH=344583e45741c457fe1862106095a5eb   -DCMAKE_C_COMPILER_LAUNCHER=ccache   -DCMAKE_CXX_COMPILER_LAUNCHER=ccache

echo "[BUILD] compile $(date -u)"
cmake --build out/android-arm64-native --target Telegram -j"$(nproc)"

set +e
cmake --build out/android-arm64-native --target apk -j"$(nproc)"
apk_rc=$?
if [ "$apk_rc" -ne 0 ]; then
  cmake --build out/android-arm64-native --target Telegram_apk -j"$(nproc)"
  apk_rc=$?
fi
set -e

if [ "$apk_rc" -ne 0 ]; then
  echo "APK target failed." >&2
  exit "$apk_rc"
fi

APK_PATH="$(find out/android-arm64-native -type f -name '*.apk' | head -n 1 || true)"
if [ -z "$APK_PATH" ]; then
  echo "APK не найден. Смотри ошибки выше в логе сборки."
  exit 1
fi

echo "APK: $APK_PATH"
cp "$APK_PATH" /content/AyuGram-dev.apk
ls -lh /content/AyuGram-dev.apk



In [ ]:
from google.colab import files
files.download('/content/AyuGram-dev.apk')


In [ ]:
%%bash
set -euxo pipefail
source /content/ayumobile_env.sh

cd /content
if [ ! -d ayumobile ]; then
  env -u HTTP_PROXY -u HTTPS_PROXY -u ALL_PROXY -u http_proxy -u https_proxy -u all_proxy \
    git clone --recursive -b dev https://github.com/Perdonus/ayumobile.git
fi

cd /content/ayumobile
env -u HTTP_PROXY -u HTTPS_PROXY -u ALL_PROXY -u http_proxy -u https_proxy -u all_proxy git fetch origin dev
env -u HTTP_PROXY -u HTTPS_PROXY -u ALL_PROXY -u http_proxy -u https_proxy -u all_proxy git reset --hard origin/dev
git submodule sync --recursive
git submodule update --init --recursive
chmod +x scripts/apply-android-submodule-patches.sh
./scripts/apply-android-submodule-patches.sh
git --no-pager log --oneline -n 5
